# London Crime Analysis

### Environment Setup

In [1]:
# Standard library
from pathlib import Path

# Third-party
import pandas as pd
import numpy as np

# Project — triggers config + logging initialization
from london_crime.config import config
from london_crime.logging_config import get_logger

# Notebook display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

logger = get_logger("notebooks.01_exploration")
logger.info("Notebook 01_exploration started")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Project root: {config.paths.project_root}")

2026-09-14 18:33:05,101 | london_crime.config | INFO | Configuration loaded from: ..\config.yaml
2026-09-14 18:33:05,105 | notebooks.01_exploration | INFO | Notebook 01_exploration started
Pandas: 3.0.5
NumPy: 2.5.3
Project root: D:\ML\Portfolio\Projects\london-crime-analysis


### Load the Combined CSV

In [2]:
raw_csv_path = config.paths.raw_data_dir / config.data["raw_filename"]
logger.info(f"Loading: {raw_csv_path.relative_to(config.paths.project_root)}")

# Load with everything as strings — preserve the raw mess
df = pd.read_csv(raw_csv_path, dtype=str, keep_default_na=False)

logger.info(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

2026-09-14 18:33:08,041 | notebooks.01_exploration | INFO | Loading: data\raw\london_crimes.csv
2026-09-14 18:33:19,441 | notebooks.01_exploration | INFO | Loaded 1,140,416 rows × 13 columns
Shape: (1140416, 13)
Rows: 1,140,416
Columns: 13


### First Look

In [3]:
# First 5 rows, all columns
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_file
0,4af53e255f9f3e69d1a295a288fe02488234842b18020e0b9c6ec10452724297,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.549537,50.815808,On or near Peel Close,E01031427,Arun 004A,Violence and sexual offences,Status update unavailable,,2025-01-metropolitan-street.csv
1,790fd102d1e83b2239b5a365ab1fccbe028e6b1c9333ae96ddd1498dd04074e5,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.212745,51.409357,On or near Supermarket,E01003423,Merton 011E,Violence and sexual offences,Unable to prosecute suspect,,2025-01-metropolitan-street.csv
2,600375a5149e92bdf80d3be0db0b31ce9b8df32091f7ab6d0d2f2058489e772a,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.211590,51.410949,On or near Richmond Avenue,E01003423,Merton 011E,Vehicle crime,Unable to prosecute suspect,,2025-01-metropolitan-street.csv
3,087d78d903f332ba60ae35ddfdd052edff68d80cd42d1f701dad07e842c0d858,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.207944,51.410039,On or near Watery Lane,E01003423,Merton 011E,Vehicle crime,Investigation complete; no suspect identified,,2025-01-metropolitan-street.csv
4,45a4272ac23f592c44c3b24f3008d308985d3f6ed3e664e861c64d0f5292ffd4,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.212700,51.410885,On or near Chatsworth Avenue,E01003423,Merton 011E,Vehicle crime,Investigation complete; no suspect identified,,2025-01-metropolitan-street.csv


### Per-Column Stats

In [4]:
# For each column: how many empty strings?
empty_counts = (df == "").sum().sort_values(ascending=False)
empty_pct = (empty_counts / len(df) * 100).round(2)

summary = pd.DataFrame({
    "empty_count": empty_counts,
    "empty_pct": empty_pct,
})
summary

,empty_count,empty_pct
Context,1140416,100.0
Last outcome category,234967,20.6
Crime ID,234967,20.6
Month,0,0.0
Reported by,0,0.0
Longitude,0,0.0
Falls within,0,0.0
Latitude,0,0.0
Location,0,0.0
LSOA name,0,0.0


In [5]:
# Confirm: empty strings vs actual NaN
print("Total rows:", len(df))
print("\nEmpty string counts (the real 'missing'):")
print((df == "").sum().sort_values(ascending=False))

# Check if Crime ID and Last outcome category emptiness overlap
both_empty = ((df["Crime ID"] == "") & (df["Last outcome category"] == "")).sum()
print(f"\nRows where BOTH Crime ID and Last outcome are empty: {both_empty:,}")
print(f"Rows where ONLY Crime ID is empty: {((df['Crime ID'] == '') & (df['Last outcome category'] != '')).sum():,}")
print(f"Rows where ONLY Last outcome is empty: {((df['Crime ID'] != '') & (df['Last outcome category'] == '')).sum():,}")

Total rows: 1140416

Empty string counts (the real 'missing'):
Context                  1140416
Last outcome category     234967
Crime ID                  234967
Month                          0
Reported by                    0
Longitude                      0
Falls within                   0
Latitude                       0
Location                       0
LSOA name                      0
LSOA code                      0
Crime type                     0
source_file                    0
dtype: int64

Rows where BOTH Crime ID and Last outcome are empty: 234,967
Rows where ONLY Crime ID is empty: 0
Rows where ONLY Last outcome is empty: 0


#### Finding: Every row that lacks a Crime ID also lacks a Last outcome category — and vice versa. They are never independently missing.

This is the signature of anonymized records. The Police.uk data documentation and research on police data quality confirm that when a crime is sensitive (e.g., domestic abuse, sexual offences, or cases where identifying the record could compromise a victim), the publisher redacts both the persistent identifier and the outcome. The record is still counted, but its identity and resolution status are withheld.



### Basic Info

In [6]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1140416 entries, 0 to 1140415
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype
---  ------                 --------------    -----
 0   Crime ID               1140416 non-null  str  
 1   Month                  1140416 non-null  str  
 2   Reported by            1140416 non-null  str  
 3   Falls within           1140416 non-null  str  
 4   Longitude              1140416 non-null  str  
 5   Latitude               1140416 non-null  str  
 6   Location               1140416 non-null  str  
 7   LSOA code              1140416 non-null  str  
 8   LSOA name              1140416 non-null  str  
 9   Crime type             1140416 non-null  str  
 10  Last outcome category  1140416 non-null  str  
 11  Context                1140416 non-null  str  
 12  source_file            1140416 non-null  str  
dtypes: str(13)
memory usage: 391.2 MB


In [7]:
# Check for constant columns (single value)
for col in ["Reported by", "Falls within"]:
    unique_vals = df[col].nunique()
    print(f"{col}: {unique_vals} unique value(s)")
    if unique_vals <= 5:
        print(df[col].value_counts())
    print()

# Cardinality of categorical-ish columns
print("Cardinality check:")
for col in ["Crime type", "LSOA code", "LSOA name", "Location"]:
    print(f"  {col}: {df[col].nunique():,} unique values")

# Check date range consistency
print(f"\nMonth range: {df['Month'].min()} → {df['Month'].max()}")
print(f"Distinct months: {df['Month'].nunique()}")
print(f"Value counts per month:")
print(df["Month"].value_counts().sort_index())

Reported by: 1 unique value(s)
Reported by
Metropolitan Police Service    1140416
Name: count, dtype: int64

Falls within: 1 unique value(s)
Falls within
Metropolitan Police Service    1140416
Name: count, dtype: int64

Cardinality check:
  Crime type: 14 unique values
  LSOA code: 8,445 unique values
  LSOA name: 8,445 unique values
  Location: 38,778 unique values

Month range: 2025-01 → 2025-12
Distinct months: 12
Value counts per month:
Month
2025-01     86952
2025-02     83660
2025-03     93571
2025-04     93211
2025-05     99386
2025-06    100914
2025-07    106677
2025-08     99050
2025-09     93280
2025-10     98545
2025-11     94209
2025-12     90961
Name: count, dtype: int64


In [8]:
from london_crime.data_cleaning import DataCleaner
cleaner = DataCleaner(df)  # re-run against the raw df
df_clean, report = cleaner.clean()
print(report.summary())
print()
print(f"Final shape: {df_clean.shape}")
print(f"Final columns: {list(df_clean.columns)}")
df_clean.dtypes

Cleaning Report
Rows:    1,140,416 → 1,140,024
Columns: 13 → 10

Dropped columns (4): Reported by, Falls within, Context, Crime ID
Derived columns: is_anonymized
Stripped prefixes: Location: 'On or near '

Notes:
  - Normalized 1,610,350 empty strings to pd.NA across 13 columns
  - Dropped 'Reported by': constant value 'Metropolitan Police Service'
  - Dropped 'Falls within': constant value 'Metropolitan Police Service'
  - Dropped 'Context': 100% empty
  - Converted 'Month' from 'YYYY-MM' string to datetime
  - Converted 'Latitude' and 'Longitude' from strings to floats
  - Stripped 'On or near ' prefix from 1,140,416 Location values
  - Converted categorical string columns to pandas 'category' dtype
  - Added 'is_anonymized' flag: 234,967 records (20.60%) are privacy-redacted
  - Dropped 392 exact duplicate rows (among rows with non-null Crime ID, keep='first')
  - Dropped 'Crime ID': not a reliable unique key (4,513 duplicate groups identified); no analytical value

Final shape: (11

Month                    datetime64[us]
Longitude                       float64
Latitude                        float64
Location                            str
LSOA code                      category
LSOA name                      category
Crime type                     category
Last outcome category               str
source_file                    category
is_anonymized                      bool
dtype: object

In [9]:
cleaned_path = config.paths.processed_data_dir / config.data["cleaned_filename"]
df_clean.to_parquet(cleaned_path, index=False)

logger.info(f"Saved cleaned data to: {cleaned_path.relative_to(config.paths.project_root)}")
print(f"Saved: {cleaned_path}")
print(f"Shape: {df_clean.shape}")
print(f"File size: {cleaned_path.stat().st_size / 1024**2:.1f} MB")

2026-09-14 18:33:55,124 | notebooks.01_exploration | INFO | Saved cleaned data to: data\processed\london_crimes_clean.parquet
Saved: D:\ML\Portfolio\Projects\london-crime-analysis\data\processed\london_crimes_clean.parquet
Shape: (1140024, 10)
File size: 10.8 MB


In [10]:
df_check = pd.read_parquet(cleaned_path)
print(f"Reloaded shape: {df_check.shape}")
print(f"Reloaded memory: {df_check.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df_check.dtypes

Reloaded shape: (1140024, 10)
Reloaded memory: 97.7 MB


Month                    datetime64[us]
Longitude                       float64
Latitude                        float64
Location                            str
LSOA code                      category
LSOA name                      category
Crime type                     category
Last outcome category               str
source_file                    category
is_anonymized                      bool
dtype: object

# EDA

In [11]:
from london_crime.eda import EDA

eda = EDA(df_clean)
results = eda.run_all()

print("=== Top 14 Crime Types ===")
print(results.crime_type_counts)
print()
print("=== Monthly Totals ===")
print(results.monthly_totals)
print()
print("=== Top 10 LSOAs ===")
print(results.top_lsoas)

2026-09-14 18:36:07,202 | london_crime.eda | INFO | Crime types: 14; top = Violence and sexual offences (270,115)
2026-09-14 18:36:07,220 | london_crime.eda | INFO | Top LSOA: Westminster 013G (13,249)
2026-09-14 18:36:07,434 | london_crime.eda | INFO | Months: 12; range 83,660-106,634
2026-09-14 18:36:07,981 | london_crime.eda | INFO | Anonymization rate by crime type computed
2026-09-14 18:36:08,117 | london_crime.eda | INFO | Top location: Supermarket (47,486)
=== Top 14 Crime Types ===
Crime type
Violence and sexual offences    270115
Anti-social behaviour           234967
Other theft                     100244
Shoplifting                      86774
Theft from the person            85420
Vehicle crime                    84205
Public order                     58054
Criminal damage and arson        54677
Drugs                            53885
Burglary                         47684
Robbery                          31332
Bicycle theft                    13826
Other crime               

In [12]:
df_clean[df_clean["Crime type"] == "Anti-social behaviour"]["is_anonymized"].sum()

np.int64(234967)

In [13]:
asb_df = df_clean[df_clean["Crime type"] == "Anti-social behaviour"]
print(f"ASB total: {len(asb_df):,}")
print(f"ASB anonymized: {asb_df['is_anonymized'].sum():,}")
print(f"ASB anonymized %: {asb_df['is_anonymized'].mean() * 100:.2f}%")
print()
print("Anonymization rate by crime type:")
print(results.anonymized_by_crime_type.round(2))

ASB total: 234,967
ASB anonymized: 234,967
ASB anonymized %: 100.00%

Anonymization rate by crime type:
is_anonymized                 False  True 
Crime type                                
Anti-social behaviour           0.0  100.0
Bicycle theft                 100.0    0.0
Burglary                      100.0    0.0
Criminal damage and arson     100.0    0.0
Drugs                         100.0    0.0
Other crime                   100.0    0.0
Other theft                   100.0    0.0
Possession of weapons         100.0    0.0
Public order                  100.0    0.0
Robbery                       100.0    0.0
Shoplifting                   100.0    0.0
Theft from the person         100.0    0.0
Vehicle crime                 100.0    0.0
Violence and sexual offences  100.0    0.0


# Visualization 

In [14]:
from london_crime.visualization import (
    plot_crime_type_counts,
    plot_monthly_trend,
    plot_top_lsoas,
    plot_anonymization_heatmap,
    plot_top_locations,
)

plot_crime_type_counts(results.crime_type_counts)
plot_monthly_trend(results.monthly_totals)
plot_top_lsoas(results.top_lsoas)
plot_anonymization_heatmap(results.anonymized_by_crime_type)
plot_top_locations(results.location_top)

import os
print("Figures saved:")
for f in sorted(os.listdir(config.paths.figures_dir)):
    print(f"  {f}")

2026-09-14 18:38:05,179 | london_crime.visualization | INFO | Saved figure: outputs\figures\crime_type_counts.png
2026-09-14 18:38:05,196 | matplotlib.category | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-09-14 18:38:05,202 | matplotlib.category | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-09-14 18:38:05,696 | london_crime.visualization | INFO | Saved figure: outputs\figures\monthly_trend.png
2026-09-14 18:38:06,280 | london_crime.visualization | INFO | Saved figure: outputs\figures\top_lsoas.png
2026-09-14 18:38:07,415 | london_crime.visualization | INFO | Saved figure: outputs\figures\anonymization_heatmap.png
2026-09-14 18:38:08,186 | london_crime.visualization | INFO | 